# 01 · Data audit

**Goal:** know what every column means *at dispatch time*, find every defect in the pack, and turn each finding into a rule for `policy.md`.

Every check below reads the **raw files as text** (no type guessing), so defects show up instead of being silently "fixed" by pandas.

| § | Check |
|---|---|
| 1 | File parsing (Excel artefacts) |
| 2 | Dates |
| 3 | Duplicates |
| 4 | Order value (paise bug) |
| 5 | Reference tables (customers, products) |
| 6 | Leakage: service events and pickups |
| 7 | Leakage: prior orders / prior returns |
| 8 | Delivery notes (leakage + PII) |
| 9 | Pincode default |
| 10 | Timezones (ops-policy §9) |
| 11 | Categories and ranges, train vs test |
| 12 | Label sanity |
| 13 | Summary → rules |

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_colwidth", 60)

DATA = Path("../data")

def read_raw(name):
    '''Read a pack file as text: no dtype guessing, empty string -> NA.'''
    return pd.read_csv(DATA / name, dtype=str, keep_default_na=False, na_values=[""])

raw = {n: read_raw(n) for n in ["train.csv", "test_unlabelled.csv", "customers.csv", "products.csv", "sample_submission.csv"]}
{n: d.shape for n, d in raw.items()}

{'train.csv': (11156, 20),
 'test_unlabelled.csv': (2097, 20),
 'customers.csv': (9001, 20),
 'products.csv': (199, 20),
 'sample_submission.csv': (2097, 20)}

## 1 · File parsing - Excel artefacts

In [2]:
for n, d in raw.items():
    unnamed = [c for c in d.columns if c.startswith("Unnamed")]
    filled = int(d[unnamed].notna().sum().sum()) if unnamed else 0
    all_blank_rows = int(d.drop(columns=unnamed).isna().all(axis=1).sum())
    print(f"{n:24s} cols={d.shape[1]:2d}  unnamed={len(unnamed):2d} (non-empty cells: {filled})  fully blank rows={all_blank_rows}")

train.csv                cols=20  unnamed= 1 (non-empty cells: 0)  fully blank rows=1
test_unlabelled.csv      cols=20  unnamed= 2 (non-empty cells: 0)  fully blank rows=1
customers.csv            cols=20  unnamed=15 (non-empty cells: 0)  fully blank rows=1
products.csv             cols=20  unnamed=14 (non-empty cells: 0)  fully blank rows=178
sample_submission.csv    cols=20  unnamed=18 (non-empty cells: 0)  fully blank rows=1


In [3]:
def tidy(d):
    '''Drop Excel artefact columns (all empty) and fully blank rows.'''
    d = d.loc[:, ~d.columns.str.startswith("Unnamed")]
    return d[~d.isna().all(axis=1)].reset_index(drop=True)

tr_raw, te_raw = tidy(raw["train.csv"]), tidy(raw["test_unlabelled.csv"])
cu, pr, ss = tidy(raw["customers.csv"]), tidy(raw["products.csv"]), tidy(raw["sample_submission.csv"])
print("train", tr_raw.shape, "| test", te_raw.shape, "| customers", cu.shape, "| products", pr.shape, "| sample", ss.shape)
print("test columns == train columns minus 'returned':", list(te_raw.columns) == [c for c in tr_raw.columns if c != "returned"])
print("sample_submission ids == test ids (same order):", ss.order_id.tolist() == te_raw.order_id.tolist())

train (11155, 19) | test (2096, 18) | customers (9000, 5) | products (21, 6) | sample (2096, 2)
test columns == train columns minus 'returned': True
sample_submission ids == test ids (same order): True


**Reading:** every file carries empty `Unnamed:*` columns and a trailing blank row (`products.csv` has 178 blank rows), which are artefacts of an Excel export. Real content: train 11,155 rows, test 2,096, customers 9,000, products 21, sample submission 2,096 (same IDs, same order as test).

**Rule:** drop `Unnamed:*` columns and fully blank rows on load.

## 2 · Dates
Explicit format `%m/%d/%Y %H:%M` (the files were saved by Excel in US format). Check failures and that day values > 12 occur, which proves month/day are not swapped.

In [4]:
FMT = "%m/%d/%Y %H:%M"
for name, d in [("train", tr_raw), ("test", te_raw)]:
    ts = pd.to_datetime(d.order_placed_at, format=FMT, errors="coerce")
    first = d.order_placed_at.str.split("/").str[0].astype(int)
    second = d.order_placed_at.str.split("/").str[1].astype(int)
    print(f"{name}: parse failures={ts.isna().sum()}  range={ts.min()} -> {ts.max()}  "
          f"max(first field)={first.max()}  max(second field)={second.max()}")
tr_raw["ts"] = pd.to_datetime(tr_raw.order_placed_at, format=FMT)
te_raw["ts"] = pd.to_datetime(te_raw.order_placed_at, format=FMT)
for c in ["signup_date"]:
    print("customers.signup_date failures:", pd.to_datetime(cu[c], format="%m/%d/%Y", errors="coerce").isna().sum())
print("products.launch_date failures:", pd.to_datetime(pr.launch_date, format="%m/%d/%Y", errors="coerce").isna().sum())
print("pickup_scheduled_at failures (non-empty only):",
      pd.to_datetime(tr_raw.pickup_scheduled_at.dropna(), format=FMT, errors="coerce").isna().sum())

train: parse failures=0  range=2025-04-01 00:22:00 -> 2026-06-30 23:08:00  max(first field)=12  max(second field)=31
test: parse failures=0  range=2026-07-01 00:51:00 -> 2026-09-30 23:46:00  max(first field)=9  max(second field)=31
customers.signup_date failures: 0
products.launch_date failures: 0
pickup_scheduled_at failures (non-empty only): 0


In [5]:
rows_per_month = (pd.concat([tr_raw.assign(set="train"), te_raw.assign(set="test")])
                  .assign(month=lambda d: d.ts.dt.to_period("M"))
                  .groupby(["set", "month"]).size().unstack(fill_value=0))
rows_per_month  # train includes duplicate copies at this point

month,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
set,,,,,,,,,,,,,,,,,,
test,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,668,733,695
train,698,708,721,785,759,747,748,737,723,769,710,789,760,759,742,0,0,0


**Reading:** zero parse failures with the explicit format. The second field reaches 31, so it is the day: month and day are not swapped. Train covers **Apr 2025 – Jun 2026** (15 months). Test covers **Jul – Sep 2026**, the *next quarter*, not October.

**Rule:** parse orders and pickups with `%m/%d/%Y %H:%M`, and reference-table dates with `%m/%d/%Y`. This runs before any month-based fix.

## 3 · Duplicates

In [6]:
dup_mask = tr_raw.order_id.duplicated(keep=False)
print("train rows:", len(tr_raw), "| unique order_id:", tr_raw.order_id.nunique(), "| extra copies:", tr_raw.order_id.duplicated().sum())
print("test rows:", len(te_raw), "| unique order_id:", te_raw.order_id.nunique())
print("order_id overlap train/test:", len(set(tr_raw.order_id) & set(te_raw.order_id)))
print("source (train):", tr_raw.source.value_counts().to_dict(), "| source (test):", te_raw.source.value_counts().to_dict())
dups = tr_raw[dup_mask]
print("copies per duplicated id:", dups.groupby("order_id").size().value_counts().to_dict())
print("source pattern in each pair:", dups.groupby("order_id").source.apply(lambda s: "+".join(sorted(s))).value_counts().to_dict())
print("sales_channel of duplicated orders:", dups.drop_duplicates("order_id").sales_channel.value_counts().to_dict())

train rows: 11155 | unique order_id: 10504 | extra copies: 651
test rows: 2096 | unique order_id: 2096
order_id overlap train/test: 0
source (train): {'crm': 10504, 'partner_feed': 651} | source (test): {'crm': 2096}
copies per duplicated id: {2: 651}
source pattern in each pair: {'crm+partner_feed': 651}
sales_channel of duplicated orders: {'partner_outlet': 552, 'marketplace': 37, 'web': 31, 'app': 31}


In [7]:
cols = [c for c in tr_raw.columns if c not in ("source", "ts")]
g = dups.groupby("order_id")
differing = {c: int((g[c].nunique(dropna=False) > 1).sum()) for c in cols}
print("pairs where a column differs between copies:", {k: v for k, v in differing.items() if v} or "none - copies are identical in every column (incl. label)")
print("return rate - duplicated orders:", round(dups.drop_duplicates('order_id').returned.astype(int).mean(), 4),
      "| all orders:", round(tr_raw.drop_duplicates('order_id').returned.astype(int).mean(), 4))

pairs where a column differs between copies: none - copies are identical in every column (incl. label)
return rate - duplicated orders: 0.1029 | all orders: 0.1142


In [8]:
# Apply rule: keep the crm copy (identical anyway)
tr = tr_raw.sort_values("source").drop_duplicates("order_id", keep="first").sort_values("ts").reset_index(drop=True)
te = te_raw.copy()
tr["returned"] = tr.returned.astype(int)
print("train after dedupe:", tr.shape, "| sources kept:", tr.source.value_counts().to_dict(), "| return rate:", round(tr.returned.mean(), 4))

train after dedupe: (10504, 20) | sources kept: {'crm': 10504} | return rate: 0.1142


**Reading:** 651 orders appear twice (`crm` + `partner_feed`), identical in every column including the label. Most are partner-outlet orders (552), but 99 are app/web/marketplace. Test has no duplicates, no `partner_feed` rows and no `order_id` overlap with train.

**Rule:** dedupe on `order_id`, keeping the `crm` copy → **10,504 orders, return rate 11.42%**.

## 4 · Order value - the October payment-gateway question
Compare `order_value_inr` with `list_price × qty × (1 − discount/100)` month by month.

In [9]:
prices = pr.set_index("sku").list_price_inr.astype(float)

def value_ratio(d):
    expected = d.sku.map(prices) * d.qty.astype(float) * (1 - d.discount_pct.astype(float) / 100)
    return d.order_value_inr.astype(float) / expected

for d in (tr, te):
    d["month"] = d.ts.dt.to_period("M")
    d["value_ratio"] = value_ratio(d)
print("unknown SKUs - train:", tr.sku.map(prices).isna().sum(), "test:", te.sku.map(prices).isna().sum())
summary = pd.concat([tr.assign(set="train"), te.assign(set="test")]).groupby(["set", "month"]).value_ratio.agg(["count", "min", "median", "max"]).round(4)
summary

unknown SKUs - train: 0 test: 0


count    min  median    max
set   month                               
test  2026-07    668    1.0     1.0    1.0
      2026-08    733    1.0     1.0    1.0
      2026-09    695    1.0     1.0    1.0
train 2025-04    661    1.0     1.0    1.0
      2025-05    674    1.0     1.0    1.0
      2025-06    678    1.0     1.0    1.0
      2025-07    736    1.0     1.0    1.0
      2025-08    714    1.0     1.0    1.0
      2025-09    707    1.0     1.0    1.0
      2025-10    700  100.0   100.0  100.0
      2025-11    698    1.0     1.0    1.0
      2025-12    684    1.0     1.0    1.0
      2026-01    719    1.0     1.0    1.0
      2026-02    664    1.0     1.0    1.0
      2026-03    743    1.0     1.0    1.0
      2026-04    705    1.0     1.0    1.0
      2026-05    722    1.0     1.0    1.0
      2026-06    699    1.0     1.0    1.0

In [10]:
oct25 = tr.month == pd.Period("2025-10")
print("Oct-2025 rows:", oct25.sum(), "| ratio exactly 100 (±0.5%):", ((tr.value_ratio[oct25] - 100).abs() < 0.5).sum())
print("Rows outside Oct-2025 with ratio off by >1%:", ((tr.value_ratio[~oct25] - 1).abs() > 0.01).sum(),
      "| test rows off by >1%:", ((te.value_ratio - 1).abs() > 0.01).sum())
print("Oct-2025 payment modes:", tr[oct25].payment_mode.value_counts().to_dict(), "(all modes affected -> gateway-wide, not one method)")
# Apply rule and re-check
tr["order_value_clean"] = tr.order_value_inr.astype(float).where(~oct25, tr.order_value_inr.astype(float) / 100)
te["order_value_clean"] = te.order_value_inr.astype(float)
chk = tr.order_value_clean / (tr.order_value_inr.astype(float) / tr.value_ratio)
print("after /100 fix: max |ratio-1| in train =", round((chk - 1).abs().max(), 6))

Oct-2025 rows: 700 | ratio exactly 100 (±0.5%): 700
Rows outside Oct-2025 with ratio off by >1%: 0 | test rows off by >1%: 0
Oct-2025 payment modes: {'prepaid_upi': 281, 'cod': 229, 'prepaid_card': 122, 'emi': 68} (all modes affected -> gateway-wide, not one method)
after /100 fix: max |ratio-1| in train = 0.0


**Reading:** **all 700 Oct-2025 orders are exactly 100× (stored in paise)**, across every payment mode. Every other month and every test row matches `list price × qty × (1 − discount)` exactly. The October gateway bug affects **training only**; the data we're scored on is clean.

**Rule:** divide Oct-2025 `order_value_inr` by 100 (verified: zero deviation after the fix). Rebuilding from list price gives the same number, so the simpler fix is preferred.

## 5 · Reference tables

In [11]:
print("customers:", cu.shape, "| duplicate ids:", cu.customer_id.duplicated().sum(), "| shield:", cu.shield_member.value_counts().to_dict())
print("products:", pr.shape, "| duplicate skus:", pr.sku.duplicated().sum())
print("orders with unknown customer - train:", (~tr.customer_id.isin(cu.customer_id)).sum(), "test:", (~te.customer_id.isin(cu.customer_id)).sum())
print("test customers never seen in train:", (~te.customer_id.isin(tr.customer_id)).sum(), "of", len(te))
pr_view = pr.copy(); pr_view["list_price_inr"] = pr_view.list_price_inr.astype(float)
pr_view

customers: (9000, 5) | duplicate ids: 0 | shield: {'N': 7001, 'Y': 1999}
products: (21, 6) | duplicate skus: 0
orders with unknown customer - train: 0 test: 0
test customers never seen in train: 660 of 2096


,sku,family,model_name,list_price_inr,warranty_months,launch_date
0,KH-AF-01,Air Fryer,Kestrel Air Fryer Lite,5199.0,12,11/4/2023
1,KH-AF-02,Air Fryer,Kestrel Air Fryer Pro,6499.0,12,9/7/2023
2,KH-AF-03,Air Fryer,Kestrel Air Fryer Max,8773.0,12,2/2/2024
3,KH-MG-01,Mixer Grinder,Kestrel Mixer Grinder Lite,3199.0,24,5/27/2023
4,KH-MG-02,Mixer Grinder,Kestrel Mixer Grinder Pro,3999.0,24,9/20/2023
5,KH-MG-03,Mixer Grinder,Kestrel Mixer Grinder Max,5398.0,24,12/9/2024
6,KH-WP-01,Water Purifier,Kestrel Water Purifier Lite,11999.0,12,9/3/2024
7,KH-WP-02,Water Purifier,Kestrel Water Purifier Pro,14999.0,12,11/19/2023
8,KH-WP-03,Water Purifier,Kestrel Water Purifier Max,20248.0,12,10/16/2023
9,KH-RV-01,Robot Vacuum,Kestrel Robot Vacuum Lite,17599.0,12,3/6/2024


In [12]:
cu["signup"] = pd.to_datetime(cu.signup_date, format="%m/%d/%Y")
pr["launch"] = pd.to_datetime(pr.launch_date, format="%m/%d/%Y")
for name, d in [("train", tr), ("test", te)]:
    m = d.merge(cu[["customer_id", "signup", "shield_member"]], on="customer_id", how="left").merge(pr[["sku", "launch"]], on="sku", how="left")
    before_signup = m.ts < m.signup
    before_launch = m.ts < m.launch
    extra = f" | return rate when before signup: {m.loc[before_signup, 'returned'].mean():.3f} vs {m.loc[~before_signup, 'returned'].mean():.3f}" if "returned" in m else ""
    print(f"{name}: orders placed BEFORE customer signup_date = {before_signup.sum()} ({before_signup.mean():.1%}) | before product launch_date = {before_launch.sum()}{extra}")
    print(f"   shield share of orders: {(m.shield_member == 'Y').mean():.3f}")

train: orders placed BEFORE customer signup_date = 1999 (19.0%) | before product launch_date = 0 | return rate when before signup: 0.113 vs 0.115
   shield share of orders: 0.222


test: orders placed BEFORE customer signup_date = 18 (0.9%) | before product launch_date = 0
   shield share of orders: 0.221


**Reading:** every order joins to a customer and a product, and every product launched before its orders. **660 of 2,096 test orders (31%) are from customers never seen in train**, so the model can't lean on customer identity (and mustn't).

**`signup_date` is unreliable, and differently so in train and test:** 19% of train orders are dated *before* the customer's own signup, against 0.9% of test orders. Return rates are the same either way (11.3% vs 11.5%).

`customers.csv` is a **snapshot**: some orders are dated before the customer's own `signup_date`. So `signup_date` (and therefore tenure) is unreliable, and `shield_member` may reflect today's status rather than the status at order time. We can't fix this from the pack, so the rule is to guard tenure and test the model's reliance on these fields with an ablation (Phase 4).

## 6 · Leakage - service events and pickups
Tanmay: *"The service and pickup columns are pulled from the service system as of export day."* README: test = *"the snapshot the warehouse sees at dispatch"*.

In [13]:
ct = pd.crosstab(tr.last_service_event_type, tr.returned, margins=True)
ct["return_rate"] = (ct[1] / ct["All"]).round(3)
print("TRAIN: last_service_event_type x returned"); display(ct)
print("TEST distribution:", te.last_service_event_type.value_counts().to_dict())
print("TRAIN distribution:", tr.last_service_event_type.value_counts().to_dict())

TRAIN: last_service_event_type x returned


returned,0,1,All,return_rate
last_service_event_type,,,,
DEMO_DONE,1417,0,1417,0.000
INSTALL_DONE,1354,0,1354,0.000
NONE,6258,278,6536,0.043
REVERSE_PICKUP,0,750,750,1.000
TECH_VISIT,275,172,447,0.385
All,9304,1200,10504,0.114


TEST distribution: {'NONE': 1553, 'INSTALL_BOOKED': 543}
TRAIN distribution: {'NONE': 6536, 'DEMO_DONE': 1417, 'INSTALL_DONE': 1354, 'REVERSE_PICKUP': 750, 'TECH_VISIT': 447}


In [14]:
tr["family_code"] = tr.sku.str[3:5]; te["family_code"] = te.sku.str[3:5]
print("TRAIN event x family"); display(pd.crosstab(tr.last_service_event_type, tr.family_code))
print("TEST event x family"); display(pd.crosstab(te.last_service_event_type, te.family_code))

TRAIN event x family


family_code,AF,CF,IC,MG,RH,RV,WP
last_service_event_type,,,,,,,
DEMO_DONE,0,531,0,0,0,448,438
INSTALL_DONE,0,486,0,0,0,426,442
NONE,1310,412,1327,1311,1334,436,406
REVERSE_PICKUP,108,69,79,56,116,193,129
TECH_VISIT,98,33,70,84,75,51,36


TEST event x family


family_code,AF,CF,IC,MG,RH,RV,WP
last_service_event_type,,,,,,,
INSTALL_BOOKED,0,185,0,0,0,175,183
NONE,304,125,288,300,289,129,118


In [15]:
has_pick = tr.pickup_scheduled_at.notna()
print("TRAIN pickup present x returned"); display(pd.crosstab(has_pick, tr.returned))
print("TEST rows with pickup_scheduled_at:", te.pickup_scheduled_at.notna().sum(), "of", len(te))
tr["pickup_ts"] = pd.to_datetime(tr.pickup_scheduled_at, format=FMT)
lag_days = (tr.pickup_ts - tr.ts).dt.total_seconds() / 86400
print("pickup - order lag (days):", lag_days.describe().round(1).to_dict())
print("pickups booked before the order was placed:", (lag_days < 0).sum())

TRAIN pickup present x returned


returned,0,1
pickup_scheduled_at,,
False,9203,70
True,101,1130


TEST rows with pickup_scheduled_at: 0 of 2096
pickup - order lag (days): {'count': 1231.0, 'mean': 11.6, 'std': 4.6, 'min': 4.0, '25%': 7.4, '50%': 11.2, '75%': 15.3, 'max': 19.4}
pickups booked before the order was placed: 0


**Reading:**
- `REVERSE_PICKUP` occurs only on returned orders. `TECH_VISIT`, `INSTALL_DONE` and `DEMO_DONE` happen after delivery.
- Train has **no** `INSTALL_BOOKED` (overwritten by later events), while test has **only** `NONE`/`INSTALL_BOOKED`, and `INSTALL_BOOKED` appears only for families that need installation (fans, robot vacuums, water purifiers). So in test it carries no information beyond product family.
- `pickup_scheduled_at` is booked 4–19 days after the order and is empty for every test row.
- Pickups without a return (cancelled pickups, ops-policy §7) and returns without a pickup both exist, so the `returned` label is the ground truth, not the pickup.

**Both columns are recorded after dispatch → banned.**

*Counts here are on deduplicated orders: `REVERSE_PICKUP` is 789 rows before dedupe and 750 after; October 2025 is 748 → 700 for the same reason.*

## 7 · Leakage - `customer_prior_orders` / `customer_prior_returns`
Are these *as of order time* (safe) or *as of export day* (leak)? Seven checks.

In [16]:
allo = pd.concat([tr.assign(set="train"), te.assign(set="test", returned=np.nan)], ignore_index=True).sort_values("ts")
allo["po"] = allo.customer_prior_orders.astype(int)
allo["prr"] = allo.customer_prior_returns.astype(int)

# Test A: prior_orders should never decrease along a customer's own orders, and should increase by >= 1 each order
allo["po_step"] = allo.groupby("customer_id").po.diff()
print("A) prior_orders step between consecutive orders of a customer:", allo.po_step.dropna().clip(-2, 3).value_counts().sort_index().to_dict())
allo["prr_step"] = allo.groupby("customer_id").prr.diff()
print("   prior_returns step between consecutive orders:", allo.prr_step.dropna().clip(-2, 3).value_counts().sort_index().to_dict())

A) prior_orders step between consecutive orders of a customer:

 {-2.0: 956, -1.0: 1219, 0.0: 1565, 1.0: 1156, 2.0: 617, 3.0: 368}
   prior_returns step between consecutive orders: {-2.0: 97, -1.0: 493, 0.0: 4727, 1.0: 466, 2.0: 73, 3.0: 25}


In [17]:
# Test B: if prior_returns counted the CURRENT order's return, returned rows would show prr one higher than history explains.
t = allo[allo.set == "train"].copy()
t["hist_ret"] = t.groupby("customer_id").returned.cumsum() - t.returned   # returns visible in the file before this order
t["gap"] = t.prr - t.hist_ret
print("B) prr - returns-seen-earlier-in-file, by label (share of rows):")
display(pd.crosstab(t.gap.clip(-2, 3), t.returned, normalize="columns").round(3))

B) prr - returns-seen-earlier-in-file, by label (share of rows):


returned,0.0,1.0
gap,,
-2.0,0.002,0.001
-1.0,0.051,0.046
0.0,0.846,0.662
1.0,0.084,0.168
2.0,0.012,0.062
3.0,0.004,0.061


In [18]:
# Test C: a customer's FIRST order in the file - prr there can only come from pre-April-2025 history.
first = t.groupby("customer_id").head(1)
print("C) first-order-in-file rows:", len(first), "| with prior_orders>0:", (first.po > 0).mean().round(3), "| with prior_returns>0:", (first.prr > 0).mean().round(3))
print("   return rate of first orders by prr>0:", first.groupby(first.prr > 0).returned.mean().round(3).to_dict())
# Test D: distribution in test (dispatch snapshot) vs train
print("D) prior_returns distribution, share of orders:")
display(pd.DataFrame({"train": t.prr.clip(0, 3).value_counts(normalize=True).sort_index(),
                      "test": allo[allo.set == "test"].prr.clip(0, 3).value_counts(normalize=True).sort_index()}).round(3))

C) first-order-in-file rows: 6131 | with prior_orders>0: 0.714 | with prior_returns>0: 0.136
   return rate of first orders by prr>0: {False: 0.094, True: 0.274}
D) prior_returns distribution, share of orders:


,train,test
prr,,
0,0.868,0.864
1,0.099,0.098
2,0.021,0.024
3,0.012,0.014


In [19]:
# Test E (decisive): if prior_returns included the current order's own return, NO returned order could have prr == 0.
ret = t[t.returned == 1]
print(f"E) returned orders with prior_returns == 0: {(ret.prr == 0).sum()} of {len(ret)} ({(ret.prr == 0).mean():.1%})")
# ...and train would carry ~11% extra '+1' mass vs the dispatch snapshot. Compare share with prr>0:
print(f"   share of orders with prr>0 - train: {(t.prr > 0).mean():.3f} | test (dispatch snapshot): {(allo[allo.set == 'test'].prr > 0).mean():.3f}")

E) returned orders with prior_returns == 0: 826 of 1200 (68.8%)
   share of orders with prr>0 - train: 0.132 | test (dispatch snapshot): 0.136


In [20]:
# Test F: share of orders with prior_returns > 0, by order month (train + test).
# If counted later (export day), OLDER orders would show MORE prior returns -> a falling trend. Flat/rising = consistent with as-of-order-time.
by_month = allo.groupby(allo.ts.dt.to_period("M")).agg(
    orders=("prr", "size"),
    share_prr_gt0=("prr", lambda s: (s > 0).mean()),
    mean_prr=("prr", "mean"),
    mean_prior_orders=("po", "mean"),
).round(3)
slope = np.polyfit(np.arange(len(by_month)), by_month.share_prr_gt0, 1)[0]
print(f"F) linear trend of share(prr>0) per month: {slope:+.4f}  (range {by_month.share_prr_gt0.min():.3f} - {by_month.share_prr_gt0.max():.3f})")
by_month.T

F) linear trend of share(prr>0) per month: -0.0002  (range 0.107 - 0.173)


ts,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07,2026-08,2026-09
orders,661.000,674.000,678.000,736.000,714.000,707.000,700.000,698.000,684.000,719.000,664.000,743.000,705.000,722.000,699.000,668.000,733.000,695.000
share_prr_gt0,0.147,0.117,0.130,0.173,0.116,0.146,0.124,0.122,0.111,0.115,0.157,0.133,0.130,0.148,0.107,0.139,0.136,0.134
mean_prr,0.209,0.150,0.170,0.257,0.164,0.214,0.170,0.169,0.158,0.160,0.211,0.167,0.170,0.211,0.150,0.198,0.190,0.197
mean_prior_orders,1.408,1.344,1.327,1.390,1.381,1.361,1.331,1.388,1.243,1.402,1.380,1.277,1.304,1.360,1.229,1.349,1.397,1.424


In [21]:
# Test G: prior_returns can never exceed prior_orders.
viol = allo[allo.prr > allo.po]
print(f"G) rows with prior_returns > prior_orders: {len(viol)} (train {int((viol.set == 'train').sum())}, test {int((viol.set == 'test').sum())})")

G) rows with prior_returns > prior_orders: 0 (train 0, test 0)


In [22]:
print("Return rate by prior_returns (train):")
display(t.groupby(t.prr.clip(0, 3)).returned.agg(["size", "mean"]).round(3))

Return rate by prior_returns (train):


,size,mean
prr,,
0,9119,0.091
1,1038,0.199
2,223,0.408
3,124,0.613


**Reading:**
- **Consistent with as-of-order-time.** The column doesn't include the order's own return: 68.8% of returned orders have `prior_returns = 0` (test E). Its distribution matches the dispatch snapshot (13.2% vs 13.6% with `prior_returns > 0`). The share is **flat across all 18 months** (test F, trend ≈ 0 per month), with no excess on older orders. **No row has `prior_returns > prior_orders`** (test G).
- **But it's a noisy CRM count.** `prior_orders` often *decreases* between a customer's consecutive orders (test A), and the counts include history outside this file (71% of customers' first orders here already have prior orders). It can't be reconstructed from the file.
- Pre-history returns are predictive on their own: first-in-file orders with `prior_returns > 0` return at 27% vs 9%.

**Decision: keep `customer_prior_orders` and `customer_prior_returns` as given, treated as a strong but imperfect signal.** Never recompute customer history from the training file (label-derived → banned).

## 8 · Delivery notes - leakage and PII

In [23]:
def note_template(s):
    if pd.isna(s): return "<none>"
    s = re.sub(r"\d+[A-Z]?", "#", s)
    return s
for d in (tr, te):
    d["note_tpl"] = d.delivery_note.map(note_template)
# Known templates = patterns that recur (>= 20 rows). Anything else is free text -> 'other' (never displayed raw).
KNOWN_TEMPLATES = sorted(t for t, n in tr.note_tpl.value_counts().items() if n >= 20 and t != "<none>")
for d in (tr, te):
    d["note_cat"] = np.where(d.note_tpl.isin(KNOWN_TEMPLATES) | (d.note_tpl == "<none>"), d.note_tpl, "other")
nt = tr.groupby("note_cat").returned.agg(["size", "mean"]).round(3).sort_values("size", ascending=False)
nt["test_rows"] = te.note_cat.value_counts()
print(len(KNOWN_TEMPLATES), "known templates")
nt

17

 known templates


,size,mean,test_rows
note_cat,,,
<none>,2589,0.121,538.0
Deliver to neighbour flat # if not home,728,0.102,123.0
Fragile - handle with care,670,0.096,125.0
"Gate code #, call before delivery",665,0.120,152.0
"Office address, weekdays only",665,0.116,148.0
"Third floor, no lift",658,0.112,144.0
Leave with security,656,0.101,133.0
"Do not call, WhatsApp only",650,0.105,129.0
Customer requested morning slot,645,0.112,133.0


In [24]:
LEAK_WORDS = r"return|refund|replace|pickup|pick up|damag|defect|broken|cancel|exchange|complain|not working"
leaky_note = tr.delivery_note.str.contains(LEAK_WORDS, case=False, na=False)
print("notes mentioning post-delivery words - train:", leaky_note.sum(),
      "| test:", te.delivery_note.str.contains(LEAK_WORDS, case=False, na=False).sum(),
      "| of which non-template ('other'):", (leaky_note & (tr.note_cat == "other")).sum())
print("notes containing gate codes (PII-like) - train:", tr.delivery_note.str.contains("Gate code", na=False).sum(),
      "| test:", te.delivery_note.str.contains("Gate code", na=False).sum())
print("phone-number-like digit runs (>=10 digits):", pd.concat([tr.delivery_note, te.delivery_note]).str.contains(r"\d{10,}", na=False).sum())

notes mentioning post-delivery words - train: 4 | test: 0 | of which non-template ('other'): 4
notes containing gate codes (PII-like) - train: 666 | test: 152
phone-number-like digit runs (>=10 digits): 0


**Reading:** notes are a small set of recurring templates, and return rates are flat across them (9–13%, apart from small landmark groups). No template describes post-delivery events. The few keyword hits are all in a handful of non-template free-text rows (bucketed as `other`, n=4, none in test). Gate codes appear in ~6% of notes and must never be shown raw.

**Decision:** raw note text is banned as a feature. The only allowed derivatives are `note_present` and `note_cat` (known template or `other`, digits removed). The UI never echoes notes.

## 9 · Pincode default
README says walk-in partner orders get `000000`. Excel stores it as the number `0`.

In [25]:
for name, d in [("train", tr), ("test", te)]:
    p = d.delivery_pincode
    print(f"{name}: length distribution={p.str.len().value_counts().to_dict()}  value '0'={int((p == '0').sum())}  value '000000'={int((p == '000000').sum())}")
print("channel mix of pincode '0' (train):", tr[tr.delivery_pincode == "0"].sales_channel.value_counts().to_dict())
print("channel mix of pincode '0' (test):", te[te.delivery_pincode == "0"].sales_channel.value_counts().to_dict())
print("return rate pincode '0' vs real:", tr.groupby(tr.delivery_pincode == "0").returned.mean().round(3).to_dict())
reg = tr.delivery_pincode.str[:3]
display(tr.groupby(reg).returned.agg(["size", "mean"]).round(3).sort_values("size", ascending=False).head(10))

train: length distribution={6: 9656, 1: 848}  value '0'=848  value '000000'=0
test: length distribution={6: 1920, 1: 176}  value '0'=176  value '000000'=0
channel mix of pincode '0' (train): {'app': 300, 'marketplace': 238, 'web': 218, 'partner_outlet': 92}
channel mix of pincode '0' (test): {'app': 54, 'marketplace': 52, 'web': 48, 'partner_outlet': 22}
return rate pincode '0' vs real: {False: 0.113, True: 0.125}


,size,mean
delivery_pincode,,
440,6425,0.128
0,848,0.125
400,581,0.086
110,581,0.093
560,574,0.094
500,532,0.092
411,525,0.072
600,438,0.062


**Reading:** the default pincode is stored as **`0`** (Excel dropped the zeros of `000000`): 848 train (8.1%) and 176 test (8.4%) rows. Contrary to the README, it is spread across **all** channels, and only 11% of these rows are partner-outlet orders. Its return rate is 12.5% vs 11.3%. Separately, region `440` (Nagpur) holds 61% of orders with a 12.8% return rate vs 6–9% elsewhere. Follow up in EDA.

**Rule:** `address_missing = (pincode == "0")`. Never treat `0` as a region. Region = first 3 digits otherwise.

## 10 · Timezones (ops-policy §9)
Legacy Zoho *resolution events* were stored in UTC and not converted. If order timestamps were affected, the hour-of-day profile before 1 Oct 2025 would be shifted by 5.5 h.

In [26]:
pre = tr.ts < "2025-10-01"
hours = pd.DataFrame({"before_1_Oct_2025": tr[pre].ts.dt.hour.value_counts(normalize=True).sort_index(),
                      "after": tr[~pre].ts.dt.hour.value_counts(normalize=True).sort_index(),
                      "test": te.ts.dt.hour.value_counts(normalize=True).sort_index()}).round(3)
print("mean order hour - before:", round(tr[pre].ts.dt.hour.mean(), 2), "after:", round(tr[~pre].ts.dt.hour.mean(), 2), "test:", round(te.ts.dt.hour.mean(), 2))
print("pickup lag (days) before vs after:", round(lag_days[pre & has_pick].median(), 2), round(lag_days[~pre & has_pick].median(), 2))
hours.T

mean order hour - before: 11.39 after: 11.4 test: 11.7


pickup lag (days) before vs after: 11.08 11.38


ts,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
before_1_Oct_2025,0.043,0.042,0.044,0.038,0.046,0.045,0.041,0.043,0.046,0.041,0.043,0.040,0.041,0.037,0.041,0.037,0.041,0.044,0.038,0.046,0.041,0.040,0.043,0.039
after,0.044,0.045,0.042,0.042,0.041,0.045,0.044,0.044,0.041,0.038,0.038,0.041,0.042,0.037,0.041,0.044,0.041,0.040,0.042,0.043,0.039,0.041,0.046,0.039
test,0.040,0.036,0.048,0.039,0.033,0.049,0.040,0.049,0.043,0.036,0.036,0.040,0.043,0.038,0.042,0.046,0.040,0.044,0.035,0.039,0.036,0.046,0.052,0.050


**Reading:** no shift. The hour-of-day profile is flat and identical before and after the 1 Oct 2025 migration (mean 11.39 vs 11.40), and pickup lags are unchanged. The §9 UTC issue affects resolution events only, and those columns are banned anyway.

**Rule:** treat order timestamps as IST; no conversion.

## 11 · Categories and ranges, train vs test

In [27]:
for c in ["sales_channel", "payment_mode", "is_gift", "qty", "promised_delivery_days", "family_code"]:
    a = tr[c].value_counts(normalize=True).round(3); b = te[c].value_counts(normalize=True).round(3)
    print(f"{c}:"); display(pd.DataFrame({"train": a, "test": b}).T)
num = ["discount_pct", "order_value_clean", "customer_prior_orders"]
display(pd.DataFrame({f"{c} ({s})": d[c].astype(float).describe().round(1) for c in num for s, d in [("train", tr), ("test", te)]}))

sales_channel:


sales_channel,app,marketplace,web,partner_outlet
train,0.341,0.273,0.263,0.123
test,0.348,0.277,0.246,0.128


payment_mode:


payment_mode,prepaid_upi,cod,prepaid_card,emi
train,0.373,0.300,0.205,0.122
test,0.373,0.307,0.211,0.109


is_gift:


...

promised_delivery_days:


promised_delivery_days,1,10,11,12,2,3,4,5,6,7,8,9
train,0.017,0.010,0.004,0.002,0.065,0.138,0.204,0.196,0.163,0.113,0.061,0.027
test,0.013,0.011,0.006,0.001,0.058,0.132,0.187,0.211,0.177,0.114,0.064,0.026


family_code:


family_code,AF,CF,IC,MG,RH,RV,WP
train,0.144,0.146,0.141,0.138,0.145,0.148,0.138
test,0.145,0.148,0.137,0.143,0.138,0.145,0.144


,discount_pct (train),discount_pct (test),order_value_clean (train),order_value_clean (test),customer_prior_orders (train),customer_prior_orders (test)
count,10504.0,2096.0,10504.0,2096.0,10504.0,2096.0
mean,10.0,10.2,8055.7,8151.2,1.3,1.4
std,7.1,7.2,7706.1,7931.9,1.3,1.3
min,0.0,0.0,879.6,1367.4,0.0,0.0
25%,5.0,5.0,2879.0,2942.1,0.0,0.0
50%,8.0,9.0,4439.6,4475.7,1.0,1.0
75%,14.0,14.0,11549.2,11549.0,2.0,2.0
max,60.0,51.0,58802.0,57020.2,10.0,9.0


In [28]:
# Shield drift: train vs test, and by order month
sh = pd.concat([tr.assign(set="train"), te.assign(set="test")]).merge(cu[["customer_id", "shield_member"]], on="customer_id", how="left")
sh["shield"] = (sh.shield_member == "Y")
print("Shield share of orders:", sh.groupby("set").shield.mean().round(3).to_dict())
print("Shield share by month:", sh.groupby(sh.ts.dt.to_period("M")).shield.mean().round(3).agg(["min", "max", "std"]).round(3).to_dict())

Shield share of orders: {'test': 0.221, 'train': 0.222}
Shield share by month: {'min': 0.193, 'max': 0.25, 'std': 0.018}


In [29]:
# Seen vs unseen customers, and first-time buyers
seen = te.customer_id.isin(tr.customer_id)
po0_te, po0_tr = te.customer_prior_orders.astype(int) == 0, tr.customer_prior_orders.astype(int) == 0
print(f"test orders from customers unseen in train: {(~seen).mean():.1%} | with prior_orders = 0: {po0_te.mean():.1%} (train: {po0_tr.mean():.1%})")
display(pd.crosstab(np.where(seen, "seen in train", "unseen in train"), np.where(po0_te, "prior_orders = 0", "prior_orders > 0"), margins=True))

test orders from customers unseen in train: 31.5% | with prior_orders = 0: 27.9% (train: 28.5%)


col_0,prior_orders = 0,prior_orders > 0,All
row_0,,,
seen in train,383,1053,1436
unseen in train,202,458,660
All,585,1511,2096


**Reading:** train and test match closely on every dispatch-time field: channel, payment, gift, qty, promised days, family, discount, order value, prior orders. Low covariate shift means validating on the most recent months should be a fair proxy for the test quarter.

| Field | Train | Test |
|---|---|---|
| Shield share of orders | 22.2% | 22.1% (monthly range 19.3–25.0%, no trend) → **no drift** |
| `prior_orders = 0` (first-time buyers per CRM) | 28.5% | 27.9% |
| Customers unseen in train | - | 31.5% |

Seen/unseen and `prior_orders = 0` are **not the same group**: 458 unseen customers have CRM history (`prior_orders > 0`), and 383 orders from customers seen in train show `prior_orders = 0` (more noise in the CRM count). Error analysis (Phase 6) reports both splits separately.

## 12 · Label sanity

In [30]:
lab = tr.groupby("month").returned.agg(["size", "mean"]).round(3)
lab.columns = ["orders", "return_rate"]
lab.T

month,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-01,2026-02,2026-03,2026-04,2026-05,2026-06
orders,661.00,674.00,678.000,736.000,714.000,707.000,700.000,698.000,684.000,719.000,664.000,743.000,705.000,722.000,699.00
return_rate,0.15,0.12,0.094,0.118,0.129,0.085,0.116,0.106,0.102,0.113,0.128,0.109,0.121,0.115,0.11


**Reading:** monthly return rate ranges from 8.5% (Sep 2025) to 15.0% (Apr 2025), with no trend. Recent months (Apr–Jun 2026) are 11–12%, close to the overall 11.4%. Nothing suggests the label definition changed at the CRM migration.

## 13 · Summary → rules for `policy.md`

| # | Finding (evidence above) | Rule |
|---|---|---|
| 1 | Excel artefacts: empty `Unnamed` columns, blank rows | Drop on load |
| 2 | Dates are `M/D/Y H:M`; train Apr 2025 – Jun 2026, test Jul – Sep 2026 | Explicit formats; parse before any month logic |
| 3 | 651 identical `crm` + `partner_feed` pairs | Dedupe on `order_id`, keep `crm` → 10,504 orders |
| 4 | Oct-2025 `order_value_inr` is ×100 (all 700 rows); test clean | ÷100 for Oct-2025 |
| 5 | `last_service_event_type`: `REVERSE_PICKUP` = 100% returns; post-delivery events; train has no `INSTALL_BOOKED` | **Banned** |
| 6 | `pickup_scheduled_at` booked 4–19 days after order; 0 in test | **Banned** |
| 7 | `prior_orders/returns`: consistent with as-of-order time (no own return, flat by month, ≤ prior_orders, matches test), but a noisy CRM count | Keep as given (strong but imperfect); never recompute from labels |
| 8 | Notes: recurring templates, flat return rates, gate codes in ~6% | Raw text banned; `note_present` + `note_cat` only; never displayed |
| 9 | Pincode `0` = no address, all channels | `address_missing` flag; region = first 3 digits otherwise |
| 10a | Shield share 22.2% train vs 22.1% test, flat by month | No drift; no action |
| 10 | `signup_date` before order in 19% of train vs 0.9% of test; `customers.csv` is a snapshot | Tenure guarded; Shield/tenure reliance tested by ablation (Phase 4) |
| 11 | No timezone shift in order times | Treat as IST |
| 12 | `source`, `order_id`, `customer_id` carry no dispatch-time meaning | Banned as features |